In [ ]:
# ===== 掩膜提取与Labelme JSON生成=====
# 功能：1. 交互式参数调试 2. 批量生成多类别掩码 3. 输出Labelme JSON文件（group_id均为null）
# 新增：种子掩码外轮廓填充（消除反光空洞），通过HSV调试器开关控制

# 第一部分：导入库
import cv2
import numpy as np
import os
import glob
import json
import time
import sys
import shutil
from pathlib import Path
from scipy.spatial import KDTree

print("✅ 掩膜提取阶段 - 库导入成功！")

# 第二部分：全局变量和辅助函数
TERMINATE_PROGRAM = False

def nothing(x):
    """空函数，用于OpenCV滑动条回调"""
    pass

def check_termination():
    """检查是否需要终止程序"""
    global TERMINATE_PROGRAM
    return TERMINATE_PROGRAM

def area_filter_connected_components(mask, min_area=70, max_area=10000):
    """
    使用连通组件分析的面积滤波器
    保留原始掩码结构，只移除面积不在指定范围内的区域
    """
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    
    filtered_mask = np.zeros_like(mask)
    total_components = num_labels - 1
    filtered_components = 0
    
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if min_area < area < max_area:
            filtered_mask[labels == i] = 255
            filtered_components += 1
    
    return filtered_mask, total_components, filtered_components

def apply_roi_mask(mask, roi_params):
    """应用ROI掩码，只保留指定区域内的内容"""
    if roi_params is None:
        return mask
    
    roi_mask = np.zeros_like(mask)
    x, y, w, h = roi_params['x'], roi_params['y'], roi_params['w'], roi_params['h']
    roi_mask[y:y+h, x:x+w] = 255
    
    return cv2.bitwise_and(mask, roi_mask)

def generate_multi_class_mask(seed_mask, root_mask, leaf_mask):
    """
    生成多类别掩码
    返回: 多类别掩码 (0:背景, 1:种子, 2:根系, 3:叶片)
    """
    multi_class_mask = np.zeros_like(seed_mask, dtype=np.uint8)
    multi_class_mask[seed_mask > 0] = 1
    multi_class_mask[root_mask > 0] = 2
    multi_class_mask[leaf_mask > 0] = 3
    return multi_class_mask

def apply_unified_morphology(mask, kernel_size=3):
    """
    统一的形态学操作，支持可变核大小
    使用与HSV调试阶段完全相同的操作顺序：开运算→膨胀→闭运算
    """
    kernel_size = kernel_size if kernel_size % 2 == 1 else kernel_size + 1
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.dilate(mask, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    
    return mask

def create_mask_from_hsv(img, hsv_params, roi_params=None, min_area=70, max_area=10000,
                         exclude_mask=None, morph_kernel_size=3, fill_holes_contour=False):
    """
    统一的掩码生成函数，逻辑与HSV调试阶段相同
    新增 fill_holes_contour：对掩码中的每个连通区域提取外轮廓并填充内部，消除空洞
    """
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    lower = np.array(hsv_params['lower'])
    upper = np.array(hsv_params['upper'])
    mask = cv2.inRange(hsv, lower, upper)
    
    if 'invert' in hsv_params and hsv_params['invert']:
        mask = cv2.bitwise_not(mask)
    
    if exclude_mask is not None:
        exclude_inverse = cv2.bitwise_not(exclude_mask)
        mask = cv2.bitwise_and(mask, exclude_inverse)
    
    if roi_params is not None:
        mask = apply_roi_mask(mask, roi_params)
    
    mask = apply_unified_morphology(mask, kernel_size=morph_kernel_size)
    
    # 新增：提取外轮廓并填充内部（消除空洞）
    if fill_holes_contour:
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        filled_mask = np.zeros_like(mask)
        cv2.fillPoly(filled_mask, contours, 255)
        mask = filled_mask
    
    if min_area > 0 or max_area < float('inf'):
        mask, _, _ = area_filter_connected_components(mask, min_area, max_area)
    
    return mask

print("✅ 辅助函数定义完成！")

# 第三部分：调试器函数（ROI选择、HSV调参、面积过滤）
def roi_selection_tuner(img_paths):
    """ROI选择调试器"""
    global TERMINATE_PROGRAM
    
    first_img = cv2.imread(img_paths[0])
    if first_img is None:
        print("错误：无法读取第一张图像")
        return None
    
    img_height, img_width = first_img.shape[:2]
    del first_img
    
    cv2.namedWindow('ROI Selection Tuner', cv2.WINDOW_NORMAL)
    
    roi_initial_x = int(img_width * 0.15)
    roi_initial_y = int(img_height * 0.15)
    roi_initial_width = int(img_width * 0.6)
    roi_initial_height = int(img_height * 0.6)
    
    cv2.createTrackbar('X', 'ROI Selection Tuner', roi_initial_x, img_width, nothing)
    cv2.createTrackbar('Y', 'ROI Selection Tuner', roi_initial_y, img_height, nothing)
    cv2.createTrackbar('Width', 'ROI Selection Tuner', roi_initial_width, img_width, nothing)
    cv2.createTrackbar('Height', 'ROI Selection Tuner', roi_initial_height, img_height, nothing)
    cv2.createTrackbar('Image', 'ROI Selection Tuner', 0, len(img_paths)-1, nothing)
    
    print("ROI选择调试器 - 多图像测试")
    print("调整ROI参数选择感兴趣区域")
    print("ESC:退出 空格:打印参数 S:保存参数 A/D:切换图像 Q:强制终止程序")
    
    current_img_idx = 0
    current_img = None
    
    while True:
        if check_termination():
            print("程序被强制终止")
            cv2.destroyAllWindows()
            sys.exit(0)
            
        try:
            x = cv2.getTrackbarPos('X', 'ROI Selection Tuner')
            y = cv2.getTrackbarPos('Y', 'ROI Selection Tuner')
            w = cv2.getTrackbarPos('Width', 'ROI Selection Tuner')
            h = cv2.getTrackbarPos('Height', 'ROI Selection Tuner')
            new_img_idx = cv2.getTrackbarPos('Image', 'ROI Selection Tuner')
        except:
            print("窗口已关闭，退出ROI选择调试")
            break
        
        if new_img_idx != current_img_idx or current_img is None:
            current_img_idx = min(new_img_idx, len(img_paths)-1)
            if current_img is not None:
                del current_img
            current_img = cv2.imread(img_paths[current_img_idx])
            if current_img is None:
                current_img = np.zeros((img_height, img_width, 3), dtype=np.uint8)
        
        img = current_img.copy()
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 3)
        
        img_name = os.path.basename(img_paths[current_img_idx])
        cv2.putText(img, f'Image: {img_name} ({current_img_idx+1}/{len(img_paths)})', 
                   (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
        cv2.putText(img, 'Press A/D to switch images', 
                   (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(img, f'ROI: X={x}, Y={y}, W={w}, H={h}', 
                   (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(img, 'Press Q to TERMINATE program', 
                   (10, 160), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        
        display_img = img
        h_display, w_display = img.shape[:2]
        if h_display > 800 or w_display > 1200:
            scale = min(800/h_display, 1200/w_display)
            new_w = int(w_display * scale)
            new_h = int(h_display * scale)
            display_img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        
        cv2.imshow('ROI Selection Tuner', display_img)
        
        key = cv2.waitKey(1) & 0xFF
        if key == 27:
            break
        elif key == ord('q') or key == ord('Q'):
            TERMINATE_PROGRAM = True
            break
        elif key == 32:
            print(f"当前ROI参数: x={x}, y={y}, w={w}, h={h}")
        elif key == ord('s'):
            roi_params = {'x': x, 'y': y, 'w': w, 'h': h}
            print(f"ROI参数已记录: {roi_params}")
        elif key == ord('a') or key == ord('A'):
            current_img_idx = max(0, current_img_idx - 1)
            cv2.setTrackbarPos('Image', 'ROI Selection Tuner', current_img_idx)
        elif key == ord('d') or key == ord('D'):
            current_img_idx = min(len(img_paths)-1, current_img_idx + 1)
            cv2.setTrackbarPos('Image', 'ROI Selection Tuner', current_img_idx)
    
    if current_img is not None:
        del current_img
    cv2.destroyAllWindows()
    
    if check_termination():
        print("程序被强制终止")
        sys.exit(0)
    
    return {'x': x, 'y': y, 'w': w, 'h': h}

def hsv_tuner_with_invert(img_paths, part_name, roi_params=None, existing_masks=None, enable_invert=True, initial_hsv=None):
    """
    HSV阈值调试器（内存优化版本）
    新增：形态学核大小滑动条、填充孔洞轮廓开关
    """
    global TERMINATE_PROGRAM
    
    first_img = cv2.imread(img_paths[0])
    if first_img is None:
        print("错误：无法读取第一张图像")
        return None, None
    
    if initial_hsv is None:
        if part_name.lower() == "seed" or "种子" in part_name:
            initial_hsv = {'H_min': 0, 'H_max': 179, 'S_min': 0, 'S_max': 100, 'V_min': 0, 'V_max': 100}
        elif part_name.lower() == "root" or "根系" in part_name:
            initial_hsv = {'H_min': 0, 'H_max': 179, 'S_min': 0, 'S_max': 80, 'V_min': 150, 'V_max': 255}
        elif part_name.lower() == "leaf" or "叶片" in part_name:
            initial_hsv = {'H_min': 35, 'H_max': 85, 'S_min': 40, 'S_max': 255, 'V_min': 40, 'V_max': 255}
        else:
            initial_hsv = {'H_min': 0, 'H_max': 179, 'S_min': 0, 'S_max': 90, 'V_min': 95, 'V_max': 255}
    
    win_title = f'HSV Tuner - {part_name}'
    cv2.namedWindow(win_title, cv2.WINDOW_NORMAL)
    
    cv2.createTrackbar('H_min', win_title, initial_hsv['H_min'], 179, nothing)
    cv2.createTrackbar('H_max', win_title, initial_hsv['H_max'], 179, nothing)
    cv2.createTrackbar('S_min', win_title, initial_hsv['S_min'], 255, nothing)
    cv2.createTrackbar('S_max', win_title, initial_hsv['S_max'], 255, nothing)
    cv2.createTrackbar('V_min', win_title, initial_hsv['V_min'], 255, nothing)
    cv2.createTrackbar('V_max', win_title, initial_hsv['V_max'], 255, nothing)
    
    invert_mask = 0
    if enable_invert:
        cv2.createTrackbar('Invert Mask', win_title, 0, 1, nothing)
    
    cv2.createTrackbar('Image', win_title, 0, len(img_paths)-1, nothing)
    cv2.createTrackbar('Morph Kernel', win_title, 3, 11, nothing)
    cv2.createTrackbar('Fill Holes (Contour)', win_title, 0, 1, nothing)  # 新增：轮廓填充开关
    
    print(f"{part_name} HSV阈值调试器 - 多图像测试")
    print("调整滑动条找到最佳的颜色范围")
    print("Morph Kernel：形态学核大小（奇数1-11），平滑边缘")
    print("Fill Holes (Contour)：对掩码轮廓内部进行填充（1开启），用于消除反光等造成的空洞")
    if enable_invert:
        print("注意：可以使用Invert Mask滑动条切换反转掩码（黑色变白色，白色变黑色）")
    if existing_masks is not None:
        print(f"注意：当前调试器会自动排除已识别的区域")
    print("ESC:退出 空格:打印参数 S:保存参数 A/D:切换图像 Q:强制终止程序")
    
    current_img_idx = 0
    current_img = None
    current_hsv = None
    del first_img
    
    while True:
        if check_termination():
            print("程序被强制终止")
            cv2.destroyAllWindows()
            sys.exit(0)
            
        try:
            h_min = cv2.getTrackbarPos('H_min', win_title)
            h_max = cv2.getTrackbarPos('H_max', win_title)
            s_min = cv2.getTrackbarPos('S_min', win_title)
            s_max = cv2.getTrackbarPos('S_max', win_title)
            v_min = cv2.getTrackbarPos('V_min', win_title)
            v_max = cv2.getTrackbarPos('V_max', win_title)
            if enable_invert:
                invert_mask = cv2.getTrackbarPos('Invert Mask', win_title)
            morph_kernel = cv2.getTrackbarPos('Morph Kernel', win_title)
            if morph_kernel % 2 == 0:
                morph_kernel += 1
            fill_holes_contour = cv2.getTrackbarPos('Fill Holes (Contour)', win_title) == 1
            new_img_idx = cv2.getTrackbarPos('Image', win_title)
        except:
            print(f"窗口已关闭，退出{part_name} HSV调试")
            break
        
        if new_img_idx != current_img_idx or current_img is None:
            current_img_idx = min(new_img_idx, len(img_paths)-1)
            if current_img is not None:
                del current_img
                del current_hsv
            current_img = cv2.imread(img_paths[current_img_idx])
            if current_img is None:
                if current_img_idx > 0:
                    current_img = cv2.imread(img_paths[current_img_idx-1])
                else:
                    current_img = np.zeros((480, 640, 3), dtype=np.uint8)
            current_hsv = cv2.cvtColor(current_img, cv2.COLOR_BGR2HSV)
        
        lower = np.array([h_min, s_min, v_min])
        upper = np.array([h_max, s_max, v_max])
        
        temp_hsv_params = {'lower': lower.tolist(), 'upper': upper.tolist()}
        if enable_invert:
            temp_hsv_params['invert'] = invert_mask == 1
        
        exclude_mask = None
        if existing_masks is not None and current_img_idx < len(existing_masks):
            exclude_mask = existing_masks[current_img_idx]
        
        mask = create_mask_from_hsv(current_img, temp_hsv_params, roi_params, min_area=0, max_area=float('inf'),
                                    exclude_mask=exclude_mask, morph_kernel_size=morph_kernel,
                                    fill_holes_contour=fill_holes_contour)
        
        if existing_masks is not None and current_img_idx < len(existing_masks):
            existing_mask = existing_masks[current_img_idx]
            combined_mask = cv2.bitwise_or(mask, existing_mask)
            display_img = current_img.copy()
            if roi_params is not None:
                x, y, w, h = roi_params['x'], roi_params['y'], roi_params['w'], roi_params['h']
                cv2.rectangle(display_img, (x, y), (x + w, y + h), (255, 0, 0), 2)
            display_img[combined_mask > 0] = [0, 0, 255]
            alpha = 0.3
            overlay = display_img.copy()
            overlay[combined_mask > 0] = [0, 0, 255]
            cv2.addWeighted(overlay, alpha, current_img, 1 - alpha, 0, display_img)
        else:
            display_img = current_img.copy()
            if roi_params is not None:
                x, y, w, h = roi_params['x'], roi_params['y'], roi_params['w'], roi_params['h']
                cv2.rectangle(display_img, (x, y), (x + w, y + h), (255, 0, 0), 2)
            display_img[mask > 0] = [0, 0, 255]
            alpha = 0.3
            overlay = display_img.copy()
            overlay[mask > 0] = [0, 0, 255]
            cv2.addWeighted(overlay, alpha, current_img, 1 - alpha, 0, display_img)
        
        mask_display = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
        result = cv2.bitwise_and(current_img, current_img, mask=mask)
        
        cv2.putText(display_img, "Original with ROI & Mask", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        cv2.putText(mask_display, "Current Mask", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        cv2.putText(result, "Result", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        
        display = np.hstack([display_img, mask_display, result])
        
        img_name = os.path.basename(img_paths[current_img_idx])
        cv2.putText(display, f'{part_name} - Image: {img_name} ({current_img_idx+1}/{len(img_paths)})', 
                   (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)
        cv2.putText(display, f'H: [{h_min}, {h_max}], S: [{s_min}, {s_max}], V: [{v_min}, {v_max}]', 
                   (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        if enable_invert:
            invert_status = "ON" if invert_mask == 1 else "OFF"
            cv2.putText(display, f'Invert Mask: {invert_status}', (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        if existing_masks is not None:
            cv2.putText(display, f'Exclude Existing Areas: ON', (10, 190), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
        cv2.putText(display, f'Morph Kernel: {morph_kernel}x{morph_kernel}', (10, 230), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        cv2.putText(display, f'Fill Holes (Contour): {"ON" if fill_holes_contour else "OFF"}', 
                   (10, 270), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        
        h_display, w_display = display.shape[:2]
        if h_display > 600 or w_display > 3200:
            scale = min(600/h_display, 3200/w_display)
            new_w = int(w_display * scale)
            new_h = int(h_display * scale)
            display = cv2.resize(display, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        
        cv2.imshow(win_title, display)
        
        key = cv2.waitKey(1) & 0xFF
        if key == 27:
            break
        elif key == ord('q') or key == ord('Q'):
            TERMINATE_PROGRAM = True
            break
        elif key == 32:
            invert_status = f", Invert: {'ON' if invert_mask == 1 else 'OFF'}" if enable_invert else ""
            exclude_status = f", Exclude Existing: {'ON' if existing_masks is not None else 'OFF'}"
            fill_status = f", Fill Contour: {'ON' if fill_holes_contour else 'OFF'}"
            print(f"当前{part_name} HSV阈值: lower={lower}, upper={upper}{invert_status}{exclude_status}{fill_status}")
            print(f"形态学核大小: {morph_kernel}x{morph_kernel}")
        elif key == ord('s'):
            hsv_params = {'lower': lower.tolist(), 'upper': upper.tolist(), 
                          'morph_kernel_size': morph_kernel, 'fill_holes_contour': fill_holes_contour}
            if enable_invert:
                hsv_params['invert'] = invert_mask == 1
            print(f"{part_name} HSV参数已记录: {hsv_params}")
        elif key == ord('a') or key == ord('A'):
            current_img_idx = max(0, current_img_idx - 1)
            cv2.setTrackbarPos('Image', win_title, current_img_idx)
        elif key == ord('d') or key == ord('D'):
            current_img_idx = min(len(img_paths)-1, current_img_idx + 1)
            cv2.setTrackbarPos('Image', win_title, current_img_idx)
    
    if current_img is not None:
        del current_img
        del current_hsv
    cv2.destroyAllWindows()
    
    if check_termination():
        print("程序被强制终止")
        sys.exit(0)
    
    final_lower = np.array([h_min, s_min, v_min])
    final_upper = np.array([h_max, s_max, v_max])
    final_morph_kernel = morph_kernel
    final_fill_holes_contour = fill_holes_contour
    
    all_masks = []
    for img_idx, img_path in enumerate(img_paths):
        img = cv2.imread(img_path)
        if img is None:
            h, w = current_img.shape[:2] if current_img is not None else (480, 640)
            mask = np.zeros((h, w), dtype=np.uint8)
        else:
            hsv_params = {'lower': final_lower.tolist(), 'upper': final_upper.tolist(), 
                          'morph_kernel_size': final_morph_kernel, 'fill_holes_contour': final_fill_holes_contour}
            if enable_invert:
                hsv_params['invert'] = invert_mask == 1
            exclude_mask = None
            if existing_masks is not None and img_idx < len(existing_masks):
                exclude_mask = existing_masks[img_idx]
            mask = create_mask_from_hsv(img, hsv_params, roi_params, min_area=0, max_area=float('inf'),
                                        exclude_mask=exclude_mask, morph_kernel_size=final_morph_kernel,
                                        fill_holes_contour=final_fill_holes_contour)
        all_masks.append(mask)
        if img is not None:
            del img
    
    hsv_params = {'lower': final_lower.tolist(), 'upper': final_upper.tolist(), 
                  'morph_kernel_size': final_morph_kernel, 'fill_holes_contour': final_fill_holes_contour}
    if enable_invert:
        hsv_params['invert'] = invert_mask == 1
    
    return hsv_params, all_masks

def area_filter_tuner_with_combined_masks(img_paths, all_masks, roi_params=None, initial_area=70, initial_max_area=7000):
    """面积过滤参数调试器 - 多图像版本，显示掩码轮廓而非矩形框，面积数字加大"""
    global TERMINATE_PROGRAM
    
    if len(all_masks) != len(img_paths):
        print(f"错误：掩码数量({len(all_masks)})与图像数量({len(img_paths)})不匹配")
        return None
    
    cv2.namedWindow('Area Filter Tuner', cv2.WINDOW_NORMAL)
    cv2.createTrackbar('Min_Area', 'Area Filter Tuner', initial_area, 5000, nothing)
    cv2.createTrackbar('Max_Area', 'Area Filter Tuner', initial_max_area, 100000, nothing)
    cv2.createTrackbar('Image', 'Area Filter Tuner', 0, len(img_paths)-1, nothing)
    
    print("面积过滤调试器 - 多图像测试")
    print("调整最小和最大面积参数，过滤噪声保留真实植株")
    print("ESC:退出 空格:打印参数 S:保存参数 A/D:切换图像 Q:强制终止程序")
    
    current_img_idx = 0
    current_img = None
    
    while True:
        if check_termination():
            print("程序被强制终止")
            cv2.destroyAllWindows()
            sys.exit(0)
            
        try:
            min_area = cv2.getTrackbarPos('Min_Area', 'Area Filter Tuner')
            max_area = cv2.getTrackbarPos('Max_Area', 'Area Filter Tuner')
            new_img_idx = cv2.getTrackbarPos('Image', 'Area Filter Tuner')
        except:
            print("窗口已关闭，退出面积过滤调试")
            break
        
        if new_img_idx != current_img_idx or current_img is None:
            current_img_idx = min(new_img_idx, len(img_paths)-1)
            if current_img is not None:
                del current_img
            current_img = cv2.imread(img_paths[current_img_idx])
            if current_img is None:
                h, w = all_masks[current_img_idx].shape
                current_img = np.zeros((h, w, 3), dtype=np.uint8)
        
        img = current_img.copy()
        mask = all_masks[current_img_idx].copy()
        
        filtered_mask, total_components, filtered_components = area_filter_connected_components(mask, min_area, max_area)
        
        # 创建显示图像（左侧：过滤前，右侧：过滤后）
        before_img = img.copy()
        after_img = img.copy()
        
        if roi_params is not None:
            x, y, w, h = roi_params['x'], roi_params['y'], roi_params['w'], roi_params['h']
            cv2.rectangle(before_img, (x, y), (x + w, y + h), (255, 0, 0), 2)
            cv2.rectangle(after_img, (x, y), (x + w, y + h), (255, 0, 0), 2)
        
        # 显示过滤前的轮廓（红色），面积数字加大到1.2，加粗为3
        contours_before, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours_before:
            area = cv2.contourArea(cnt)
            if area > 0:
                cv2.drawContours(before_img, [cnt], -1, (0, 0, 255), 2)
                M = cv2.moments(cnt)
                if M["m00"] != 0:
                    cx = int(M["m10"] / M["m00"])
                    cy = int(M["m01"] / M["m00"])
                else:
                    x, y, w, h = cv2.boundingRect(cnt)
                    cx, cy = x + w//2, y + h//2
                cv2.putText(before_img, f'{int(area)}', (cx-30, cy-15), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 3)
        
        # 显示过滤后的轮廓（绿色），面积数字加大
        contours_after, _ = cv2.findContours(filtered_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours_after:
            area = cv2.contourArea(cnt)
            if area > 0:
                cv2.drawContours(after_img, [cnt], -1, (0, 255, 0), 2)
                M = cv2.moments(cnt)
                if M["m00"] != 0:
                    cx = int(M["m10"] / M["m00"])
                    cy = int(M["m01"] / M["m00"])
                else:
                    x, y, w, h = cv2.boundingRect(cnt)
                    cx, cy = x + w//2, y + h//2
                cv2.putText(after_img, f'{int(area)}', (cx-30, cy-15), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 3)
        
        display = np.hstack([before_img, after_img])
        h_display, w_display = display.shape[:2]
        max_height = 800
        max_width = 2400
        if h_display > max_height or w_display > max_width:
            scale = min(max_height/h_display, max_width/w_display)
            new_w = int(w_display * scale)
            new_h = int(h_display * scale)
            display_resized = cv2.resize(display, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        else:
            display_resized = display.copy()
            new_h, new_w = h_display, w_display
            scale = 1.0
        
        final_display = display_resized.copy()
        font_scale_base = new_w / 2400
        font_scale_large = max(1.0, 1.2 * font_scale_base)
        font_scale_medium = max(0.8, 1.0 * font_scale_base)
        font_scale_small = max(0.6, 0.8 * font_scale_base)
        line_thickness_base = max(1, int(2 * font_scale_base))
        
        img_name = os.path.basename(img_paths[current_img_idx])
        y_offset = 40
        y_increment = 40
        cv2.putText(final_display, f'Image: {img_name} ({current_img_idx+1}/{len(img_paths)})', 
                   (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, font_scale_large, (255,255,255), line_thickness_base)
        y_offset += y_increment
        cv2.putText(final_display, 'Press A/D to switch images', 
                   (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, font_scale_medium, (255,255,255), line_thickness_base-1)
        y_offset += y_increment
        cv2.putText(final_display, 'Press Q to TERMINATE program', 
                   (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, font_scale_medium, (0,0,255), line_thickness_base-1)
        y_offset += y_increment
        cv2.putText(final_display, f'Min Area: {min_area}', 
                   (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, font_scale_medium, (255,255,255), line_thickness_base-1)
        y_offset += y_increment
        cv2.putText(final_display, f'Max Area: {max_area}', 
                   (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, font_scale_medium, (255,255,255), line_thickness_base-1)
        y_offset += y_increment
        cv2.putText(final_display, f'Components: {filtered_components}/{total_components}', 
                   (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, font_scale_medium, (255,255,255), line_thickness_base-1)
        y_offset += y_increment
        cv2.putText(final_display, f'Area Type: Mask Region (pixels)', 
                   (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, font_scale_small, (255,255,255), line_thickness_base-1)
        
        separator_x = int(before_img.shape[1] * scale)
        cv2.line(final_display, (separator_x, 0), (separator_x, new_h), (255,255,255), 2)
        left_label_x = 10
        right_label_x = separator_x + 10
        y_offset += y_increment
        cv2.putText(final_display, 'Before Filter (Red)', (left_label_x, y_offset), 
                   cv2.FONT_HERSHEY_SIMPLEX, font_scale_small, (0,0,255), line_thickness_base-1)
        cv2.putText(final_display, 'After Filter (Green)', (right_label_x, y_offset), 
                   cv2.FONT_HERSHEY_SIMPLEX, font_scale_small, (0,255,0), line_thickness_base-1)
        
        cv2.imshow('Area Filter Tuner', final_display)
        
        key = cv2.waitKey(1) & 0xFF
        if key == 27:
            break
        elif key == ord('q') or key == ord('Q'):
            TERMINATE_PROGRAM = True
            break
        elif key == 32:
            print(f"当前面积范围: {min_area} - {max_area}, 组件数: {filtered_components}/{total_components}")
        elif key == ord('s'):
            print(f"面积参数已记录: min_area={min_area}, max_area={max_area}")
        elif key == ord('a') or key == ord('A'):
            current_img_idx = max(0, current_img_idx - 1)
            cv2.setTrackbarPos('Image', 'Area Filter Tuner', current_img_idx)
        elif key == ord('d') or key == ord('D'):
            current_img_idx = min(len(img_paths)-1, current_img_idx + 1)
            cv2.setTrackbarPos('Image', 'Area Filter Tuner', current_img_idx)
    
    if current_img is not None:
        del current_img
    cv2.destroyAllWindows()
    
    if check_termination():
        print("程序被强制终止")
        sys.exit(0)
    
    return {'min_area': min_area, 'max_area': max_area}

print("✅ 调试器函数定义完成！")

# 第四部分：Labelme JSON生成函数（无group_id关联）
def extract_components_with_centroids_sorted(mask, min_area=10, label="seed"):
    """
    提取组件并按空间位置排序，用于JSON生成
    """
    components = []
    if np.sum(mask) == 0:
        return components
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for i, contour in enumerate(contours):
        area = cv2.contourArea(contour)
        if area < min_area:
            continue
        M = cv2.moments(contour)
        if M["m00"] != 0:
            centroid_x = float(M["m10"] / M["m00"])
            centroid_y = float(M["m01"] / M["m00"])
        else:
            x, y, w, h = cv2.boundingRect(contour)
            centroid_x = float(x + w/2)
            centroid_y = float(y + h/2)
        
        epsilon = 0.006 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)
        points = []
        for point in approx:
            x, y = point[0]
            points.append([float(x), float(y)])
        
        if len(points) >= 3:
            components.append({
                'id': i + 1,
                'centroid': (centroid_x, centroid_y),
                'area': area,
                'points': points,
                'bounding_box': cv2.boundingRect(contour)
            })
    
    # 排序策略（保持原样，不影响group_id）
    if label == "seed":
        components = grid_based_sorting(components)
    elif label == "root":
        components.sort(key=lambda c: c['area'], reverse=True)
    elif label == "leaf":
        components.sort(key=lambda c: c['centroid'][1])
    else:
        components.sort(key=lambda c: (c['centroid'][1], c['centroid'][0]))
    
    for idx, component in enumerate(components, 1):
        component['id'] = idx
    
    return components

def grid_based_sorting(components, grid_rows=None, grid_cols=None, tolerance_factor=0.2):
    """基于网格的排序方法（用于种子排序）"""
    if not components:
        return components
    
    centroids = [c['centroid'] for c in components]
    x_coords = [c[0] for c in centroids]
    y_coords = [c[1] for c in centroids]
    
    if grid_rows is None or grid_cols is None:
        rows, cols = auto_detect_grid(centroids, tolerance_factor)
    else:
        rows = grid_rows
        cols = grid_cols
    
    if rows <= 1 or cols <= 1 or len(components) != rows * cols:
        components.sort(key=lambda c: (c['centroid'][1], c['centroid'][0]))
        return components
    
    x_bins = np.linspace(min(x_coords), max(x_coords), cols)
    y_bins = np.linspace(min(y_coords), max(y_coords), rows)
    
    for comp in components:
        x, y = comp['centroid']
        col_idx = np.argmin(np.abs(x_bins - x))
        row_idx = np.argmin(np.abs(y_bins - y))
        comp['grid_row'] = row_idx
        comp['grid_col'] = col_idx
        comp['grid_score'] = row_idx * cols + col_idx
    
    components.sort(key=lambda c: (c['grid_row'], c['grid_col']))
    return components

def auto_detect_grid(centroids, tolerance_factor=0.2):
    """自动检测网格的行数和列数"""
    x_coords = [c[0] for c in centroids]
    y_coords = [c[1] for c in centroids]
    sorted_x = sorted(x_coords)
    sorted_y = sorted(y_coords)
    x_diffs = np.diff(sorted_x)
    y_diffs = np.diff(sorted_y)
    if len(x_diffs) == 0 or len(y_diffs) == 0:
        return 1, len(centroids)
    avg_x_diff = np.mean(x_diffs)
    avg_y_diff = np.mean(y_diffs)
    from scipy.cluster.hierarchy import fclusterdata
    x_clusters = fclusterdata(np.array(x_coords).reshape(-1, 1), avg_x_diff * tolerance_factor, criterion='distance')
    y_clusters = fclusterdata(np.array(y_coords).reshape(-1, 1), avg_y_diff * tolerance_factor, criterion='distance')
    cols = len(np.unique(x_clusters))
    rows = len(np.unique(y_clusters))
    return rows, cols

def mask_to_labelme_json_with_grouping(image_path, multi_class_mask, output_json_path, min_contour_area=10):
    """
    将多类别掩码转换为Labelme格式的JSON文件（无group_id关联，所有形状独立）
    """
    img = cv2.imread(image_path)
    if img is not None:
        height, width = img.shape[:2]
    else:
        height, width = multi_class_mask.shape[:2]
    
    class_mapping = {1: "seed", 2: "root", 3: "leaf"}
    all_components = {}
    for class_id, class_name in class_mapping.items():
        class_mask = (multi_class_mask == class_id).astype(np.uint8) * 255
        components = extract_components_with_centroids_sorted(class_mask, min_area=min_contour_area, label=class_name)
        all_components[class_name] = components
    
    shapes = []
    # 种子
    for seed in all_components.get('seed', []):
        shapes.append({
            "label": "seed",
            "points": seed['points'],
            "group_id": None,
            "shape_type": "polygon",
            "flags": {}
        })
    # 根系
    for root in all_components.get('root', []):
        shapes.append({
            "label": "root",
            "points": root['points'],
            "group_id": None,
            "shape_type": "polygon",
            "flags": {}
        })
    # 叶片
    for leaf in all_components.get('leaf', []):
        shapes.append({
            "label": "leaf",
            "points": leaf['points'],
            "group_id": None,
            "shape_type": "polygon",
            "flags": {}
        })
    
    json_data = {
        "version": "5.1.1",
        "flags": {},
        "shapes": shapes,
        "imagePath": os.path.basename(image_path),
        "imageData": None,
        "imageHeight": height,
        "imageWidth": width
    }
    
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(json_data, f, ensure_ascii=False, indent=2)
    
    seed_count = len(all_components.get('seed', []))
    root_count = len(all_components.get('root', []))
    leaf_count = len(all_components.get('leaf', []))
    print(f"  ✓ 生成Labelme JSON: {os.path.basename(output_json_path)}")
    print(f"     种子: {seed_count}, 根系: {root_count}, 叶片: {leaf_count}（无group_id关联）")
    return json_data

def save_colored_mask(multi_class_mask, output_path):
    """
    保存多类别掩码图像，使用不同亮度表示不同部位（灰度图）
    种子(1): 200 (亮灰)
    根系(2): 100 (中灰)
    叶片(3): 50 (暗灰)
    背景: 0
    """
    h, w = multi_class_mask.shape
    gray_mask = np.zeros((h, w), dtype=np.uint8)
    gray_mask[multi_class_mask == 1] = 200   # 种子 - 亮灰
    gray_mask[multi_class_mask == 2] = 100   # 根系 - 中灰
    gray_mask[multi_class_mask == 3] = 50    # 叶片 - 暗灰
    cv2.imwrite(output_path, gray_mask)

def save_result_with_contours(image_path, seed_mask, root_mask, leaf_mask, output_path):
    """
    生成原图与带彩色轮廓的拼接图
    左侧：原图
    右侧：原图+轮廓（种子红色、根系绿色、叶片蓝色），线宽加粗至4
    """
    img = cv2.imread(image_path)
    if img is None:
        print(f"无法读取图像 {image_path}，跳过保存结果图")
        return
    img_with_contours = img.copy()
    
    # 找种子轮廓并绘制红色，线宽4
    contours_seed, _ = cv2.findContours(seed_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(img_with_contours, contours_seed, -1, (0, 0, 255), 4)
    
    # 找根系轮廓并绘制绿色，线宽4
    contours_root, _ = cv2.findContours(root_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(img_with_contours, contours_root, -1, (0, 255, 0), 4)
    
    # 找叶片轮廓并绘制蓝色，线宽4
    contours_leaf, _ = cv2.findContours(leaf_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(img_with_contours, contours_leaf, -1, (255, 0, 0), 4)
    
    # 水平拼接
    h, w = img.shape[:2]
    result = np.hstack([img, img_with_contours])
    cv2.imwrite(output_path, result)

print("✅ Labelme JSON生成函数及辅助输出函数定义完成！")

# 第五部分：批量生成函数
def batch_generate_masks_and_json_with_grouping(input_folder, output_root, all_hsv_params, 
                                               area_params, roi_params=None, debugged_parts=None):
    """批量生成Labelme JSON文件、灰度掩码图、结果拼接图，并保存到子文件夹中"""
    global TERMINATE_PROGRAM
    
    image_patterns = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tiff", "*.tif"]
    image_files = []
    for pattern in image_patterns:
        image_files.extend(glob.glob(os.path.join(input_folder, pattern)))
    
    if not image_files:
        print(f"错误：在文件夹 {input_folder} 中没有找到图片")
        return None
    
    print(f"找到 {len(image_files)} 张图片")
    
    # 创建输出子文件夹
    json_dir = os.path.join(output_root, "json")
    mask_dir = os.path.join(output_root, "mask")
    result_dir = os.path.join(output_root, "result")
    for d in [json_dir, mask_dir, result_dir]:
        os.makedirs(d, exist_ok=True)
    
    min_area = area_params['min_area']
    max_area = area_params['max_area']
    
    start_time = time.perf_counter()
    print("\n开始批量生成JSON文件、掩码图像及结果拼接图...")
    if debugged_parts is not None:
        print(f"调试的部位: {debugged_parts}")
    
    for idx, img_path in enumerate(image_files, 1):
        if check_termination():
            print("程序被强制终止")
            sys.exit(0)
        
        img_start_time = time.perf_counter()
        img = cv2.imread(img_path)
        if img is None:
            print(f"警告：无法读取图像 {img_path}")
            continue
        
        filename = os.path.basename(img_path)
        name = os.path.splitext(filename)[0]
        print(f"[{idx}/{len(image_files)}] 处理: {filename}")
        
        if debugged_parts is None:
            debugged_parts = []
            for i, params in enumerate(all_hsv_params):
                if params is not None:
                    if i == 0: debugged_parts.append("seed")
                    elif i == 1: debugged_parts.append("root")
                    elif i == 2: debugged_parts.append("leaf")
        
        # 种子掩码
        if "seed" in debugged_parts:
            seed_idx = 0
            seed_hsv = all_hsv_params[seed_idx]
            morph_kernel = seed_hsv.get('morph_kernel_size', 3)
            fill_contour = seed_hsv.get('fill_holes_contour', False)
            seed_mask = create_mask_from_hsv(img, seed_hsv, roi_params, min_area, max_area,
                                             exclude_mask=None, morph_kernel_size=morph_kernel,
                                             fill_holes_contour=fill_contour)
        else:
            h, w = img.shape[:2]
            seed_mask = np.zeros((h, w), dtype=np.uint8)
        
        # 根系掩码
        if "root" in debugged_parts:
            root_idx = 1
            root_hsv = all_hsv_params[root_idx]
            morph_kernel = root_hsv.get('morph_kernel_size', 3)
            # 根系通常不需要填充空洞，但也可以从参数中读取（默认False）
            root_mask = create_mask_from_hsv(img, root_hsv, roi_params, min_area, max_area,
                                             exclude_mask=seed_mask, morph_kernel_size=morph_kernel,
                                             fill_holes_contour=False)
        else:
            root_mask = np.zeros_like(seed_mask, dtype=np.uint8)
        
        # 叶片掩码
        if "leaf" in debugged_parts:
            leaf_idx = 2
            leaf_hsv = all_hsv_params[leaf_idx]
            morph_kernel = leaf_hsv.get('morph_kernel_size', 3)
            exclude_mask = None
            if "seed" in debugged_parts:
                exclude_mask = seed_mask
            if "root" in debugged_parts:
                if exclude_mask is not None:
                    exclude_mask = cv2.bitwise_or(exclude_mask, root_mask)
                else:
                    exclude_mask = root_mask
            leaf_mask = create_mask_from_hsv(img, leaf_hsv, roi_params, min_area, max_area,
                                             exclude_mask=exclude_mask, morph_kernel_size=morph_kernel,
                                             fill_holes_contour=False)
        else:
            leaf_mask = np.zeros_like(seed_mask, dtype=np.uint8)
        
        multi_class_mask = generate_multi_class_mask(seed_mask, root_mask, leaf_mask)
        
        # 保存 JSON 到 json 子文件夹
        json_path = os.path.join(json_dir, f"{name}.json")
        mask_to_labelme_json_with_grouping(img_path, multi_class_mask, json_path, 
                                          min_contour_area=min_area)
        
        # 保存灰度掩码图像到 mask 子文件夹
        mask_img_path = os.path.join(mask_dir, f"{name}_mask.png")
        save_colored_mask(multi_class_mask, mask_img_path)
        
        # 保存结果拼接图到 result 子文件夹
        result_img_path = os.path.join(result_dir, f"{name}_result.png")
        save_result_with_contours(img_path, seed_mask, root_mask, leaf_mask, result_img_path)
        
        del img, multi_class_mask, seed_mask, root_mask, leaf_mask
        
        now = time.perf_counter()
        elapsed = now - start_time
        avg = elapsed / idx
        eta = avg * (len(image_files) - idx)
        print(f"  耗时: {now-img_start_time:.2f}s, 平均: {avg:.2f}s, ETA: {eta:.2f}s")
    
    total_time = time.perf_counter() - start_time
    print(f"\n✅ 批量处理完成!")
    print(f"总耗时: {total_time:.2f}s")
    print(f"平均每张: {total_time/len(image_files):.2f}s")
    print(f"\n输出目录:")
    print(f"  JSON文件: {json_dir}/")
    print(f"  掩码图像: {mask_dir}/")
    print(f"  结果拼接图: {result_dir}/")
    
    return {"json_folder": json_dir, "mask_folder": mask_dir, "result_folder": result_dir, "total_images": len(image_files)}

def save_parameters(params, output_folder):
    """保存所有参数到JSON文件"""
    params_file = os.path.join(output_folder, 'mask_generation_params.json')
    with open(params_file, 'w', encoding='utf-8') as f:
        json.dump(params, f, indent=4, ensure_ascii=False)
    print(f"参数已保存到: {params_file}")
    return params_file

# 第六部分：主程序
def phase1_main():
    global TERMINATE_PROGRAM
    
    print("=" * 60)
    print("第一阶段：掩膜提取与Labelme JSON生成（无group_id关联，独立实例）")
    print("=" * 60)
    print("注意：在任何调试窗口中按 Q 键可以强制终止程序")
    print("HSV调试器中可调整形态学核大小（Morph Kernel）和轮廓填充开关（Fill Holes (Contour)）")
    
    try:
        print("\n=== 目录设置 ===")
        print("现在支持分别设置测试文件夹和批量处理文件夹！")
        test_folder = input("请输入参数调试用的测试图像文件夹路径: ").strip()
        test_folder = test_folder.strip('"').strip("'")
        if not os.path.exists(test_folder):
            print(f"错误：测试文件夹不存在 {test_folder}")
            return
        
        process_folder = input("请输入批量处理的图像文件夹路径: ").strip()
        process_folder = process_folder.strip('"').strip("'")
        if not os.path.exists(process_folder):
            print(f"错误：批量处理文件夹不存在 {process_folder}")
            return
        
        print(f"\n目录设置完成:")
        print(f"测试文件夹: {test_folder}")
        print(f"批量处理文件夹: {process_folder}")
        
        output_root = os.path.join(process_folder, "mask_generation_output")
        os.makedirs(output_root, exist_ok=True)
        
        image_patterns = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tiff", "*.tif"]
        test_image_paths = []
        for pattern in image_patterns:
            test_image_paths.extend(glob.glob(os.path.join(test_folder, pattern)))
        if not test_image_paths:
            print(f"错误：在测试文件夹中未找到任何图像文件")
            return
        
        print(f"\n在测试文件夹中找到 {len(test_image_paths)} 张图像")
        for i, path in enumerate(test_image_paths[:5]):
            print(f"  {i+1:2d}. {os.path.basename(path)}")
        if len(test_image_paths) > 5:
            print(f"  ... 还有 {len(test_image_paths)-5} 张")
        
        process_image_paths = []
        for pattern in image_patterns:
            process_image_paths.extend(glob.glob(os.path.join(process_folder, pattern)))
        if not process_image_paths:
            print(f"错误：在批量处理文件夹中未找到任何图像文件")
            return
        print(f"在批量处理文件夹中找到 {len(process_image_paths)} 张图像")
        
        print("\n开始使用测试文件夹进行参数调试...")
        
        # ROI选择
        print("\n=== ROI区域选择 ===")
        use_roi = input("是否使用ROI区域限制？(y/n): ").strip().lower()
        roi_params = None
        if use_roi in ['y', 'yes', '是']:
            print("\n启动ROI选择调试器...")
            roi_params = roi_selection_tuner(test_image_paths)
            if roi_params is None:
                print("ROI参数调试失败")
                return
            print(f"ROI参数: {roi_params}")
        
        # HSV参数调试
        print("\n=== HSV参数调试 ===")
        hsv_params_by_part = {}
        all_masks = []
        
        print("\n1. 种子HSV阈值调试...")
        hsv_params_seed, masks_seed = hsv_tuner_with_invert(test_image_paths, "seed", roi_params=roi_params, enable_invert=True)
        if hsv_params_seed is None:
            print("种子HSV参数调试失败")
            return
        hsv_params_by_part["seed"] = hsv_params_seed
        all_masks = masks_seed
        
        print("\n=== 根系HSV阈值调试 ===")
        use_root = input("是否需要调试根系HSV参数？(y/n): ").strip().lower()
        hsv_params_root = None
        if use_root in ['y', 'yes', '是']:
            print("\n2. 根系HSV阈值调试...（自动排除种子区域）")
            hsv_params_root, masks_root = hsv_tuner_with_invert(test_image_paths, "root", roi_params=roi_params, 
                                                                existing_masks=all_masks, enable_invert=False)
            if hsv_params_root is not None:
                hsv_params_by_part["root"] = hsv_params_root
                for i in range(len(all_masks)):
                    all_masks[i] = cv2.bitwise_or(all_masks[i], masks_root[i])
        
        print("\n=== 叶片HSV阈值调试 ===")
        use_leaf = input("是否需要调试叶片HSV参数？(y/n): ").strip().lower()
        hsv_params_leaf = None
        if use_leaf in ['y', 'yes', '是']:
            print("\n3. 叶片HSV阈值调试...（自动排除种子+根系区域）")
            hsv_params_leaf, masks_leaf = hsv_tuner_with_invert(test_image_paths, "leaf", roi_params=roi_params,
                                                                existing_masks=all_masks, enable_invert=False)
            if hsv_params_leaf is not None:
                hsv_params_by_part["leaf"] = hsv_params_leaf
                for i in range(len(all_masks)):
                    all_masks[i] = cv2.bitwise_or(all_masks[i], masks_leaf[i])
        
        # 构建参数列表
        all_hsv_params = []
        debugged_parts = []
        for part_name, hsv_params in [("seed", hsv_params_seed), ("root", hsv_params_root), ("leaf", hsv_params_leaf)]:
            if hsv_params is not None:
                all_hsv_params.append(hsv_params)
                debugged_parts.append(part_name)
            else:
                all_hsv_params.append(None)
        
        print("\n4. 面积过滤参数调试...")
        area_params = area_filter_tuner_with_combined_masks(test_image_paths, all_masks, roi_params)
        if area_params is None:
            print("面积参数调试失败")
            return
        
        print("\n=== 最终参数 ===")
        print(f"调试的部位: {debugged_parts}")
        part_names = {"seed":"种子", "root":"根系", "leaf":"叶片"}
        for i, hsv_params in enumerate(all_hsv_params):
            if hsv_params is not None:
                part_name = part_names.get(["seed","root","leaf"][i], f"部位{i+1}")
                invert_status = f", 反转掩码: {'是' if hsv_params.get('invert', False) else '否'}" if i==0 else ""
                exclude_status = f", 排除已识别区域: {'是' if i>0 else '否'}"
                morph_info = f", 形态学核大小: {hsv_params.get('morph_kernel_size', 3)}x{hsv_params.get('morph_kernel_size', 3)}"
                fill_status = f", 轮廓填充: {'是' if hsv_params.get('fill_holes_contour', False) else '否'}" if i==0 else ""
                print(f"  {part_name}: lower={hsv_params['lower']}, upper={hsv_params['upper']}{invert_status}{exclude_status}{morph_info}{fill_status}")
        print(f"面积参数: {area_params}")
        if roi_params is not None:
            print(f"ROI参数: {roi_params}")
        
        all_params = {
            'hsv_params_by_part': hsv_params_by_part,
            'all_hsv_params': all_hsv_params,
            'debugged_parts': debugged_parts,
            'area_params': area_params,
            'roi_params': roi_params,
            'test_image_count': len(test_image_paths),
            'process_image_count': len(process_image_paths),
            'test_folder': test_folder,
            'process_folder': process_folder,
            'output_root': output_root,
            'timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
        }
        save_parameters(all_params, output_root)
        
        print("\n=== 批量生成JSON、掩码图像及结果图 ===")
        print(f"准备为批量处理文件夹中的 {len(process_image_paths)} 张图像生成输出文件")
        print(f"输出根目录: {output_root}")
        batch_proceed = input("\n是否开始批量生成？(y/n): ").strip().lower()
        if batch_proceed in ['y', 'yes', '是']:
            print("\n开始批量处理...")
            results = batch_generate_masks_and_json_with_grouping(
                process_folder, output_root, all_hsv_params, area_params, roi_params,
                debugged_parts
            )
            if results:
                print(f"\n✅ 第一阶段完成！")
                print(f"请在Labelme中打开 {results['json_folder']} 文件夹进行人工矫正。")
                print(f"矫正完成后，运行第二阶段进行表型提取。")
        else:
            print("跳过批量生成")
        
        print("\n第一阶段程序结束")
        
    except KeyboardInterrupt:
        print("\n\n程序被用户中断")
        TERMINATE_PROGRAM = True
    except Exception as e:
        print(f"\n程序发生错误: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    phase1_main()